In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir("../")

In [3]:
from ease_recommender import *
from npmi_recommender import *

import pickle as p

In [4]:
def create_mat(row, col, bool_to_int=True):
    # bool_to_int won't count duplicates in the same row, creates a different weighting basically
    if bool_to_int:
        data = np.ones_like(row, dtype=bool)
        return csr_matrix((data, (row, col))).astype(np.int64)
    else:
        data = np.ones_like(row, dtype=np.int64)
        return csr_matrix((data, (row, col)))

def check_if_all_terms_in_str(q, terms):
    for term in terms:
        if term not in q:
            return False

    return True

def get_cat2idx(category_type, D):
    if category_type == "track":
        return D["track2idx"]
    elif category_type == "album":
        return D["album2idx"]
    elif category_type == "artist":
        return D["artist2idx"]
    else:
        raise NotImplementedError

def find_match_using_terms(terms, cat2idx):
    matches = []
    for name in cat2idx.keys():
        if check_if_all_terms_in_str(name, terms):
            matches.append(name)

    if len(matches) > 1:
        raise Exception("Multiple matches found, filter down to a single match", matches)

    return matches[0]

In [5]:
print("loading cache data...")
D = p.load(open("cached_data/spotify_preprocessed.p", "rb"))

print("building csr matrices...")

# TODO: finish implementing track and album level recommendations

# track_mat = create_mat(D["playlist_indices"], D["track_indices"])
# album_mat = create_mat(D["playlist_indices"], D["album_indices"])
artist_mat = create_mat(D["playlist_indices"], D["artist_indices"])

print("done")

loading cache data...
building csr matrices...
done


In [6]:
cat2idx = get_cat2idx("artist", D)
idx2cat = {v:k for k, v in cat2idx.items()}

In [7]:
def get_item_idx(terms):
    assert type(terms) == list
    
    a_name = find_match_using_terms(terms, cat2idx)
    a = cat2idx[a_name]
    
    print("Artist:", a_name)
    print("*" * 20)
    
    assert artist_mat[:, a].sum() > 0
    
    return a

In [8]:
mat = artist_mat
mat = csr_array(mat)

In [9]:
mat.shape

(1000000, 295860)

In [10]:
X = mat.T @ mat

In [11]:
X.shape

(295860, 295860)

In [115]:
import numpy as np
from scipy import sparse
from scipy.sparse.linalg import svds
from scipy.optimize import minimize

class SppmiUltimateCache:
    """High-performance cache matching the exact structural properties of Version 7."""
    def __init__(self, X, svd_components=32):
        self.X_csc = X.tocsc() if not sparse.isspmatrix_csc(X) else X
        self.rows, self.cols = self.X_csc.shape
        self.N_raw = self.X_csc.sum()
        self.row_sums_raw = np.array(self.X_csc.sum(axis=1)).flatten()
        self.col_sums_raw = np.array(self.X_csc.sum(axis=0)).flatten()
        
        # High-precision CSR index pointer lookup for context fertility
        X_csr = self.X_csc.tocsr()
        self.fertility_raw = np.diff(X_csr.indptr).astype(np.float32)
        self.fertility_norm = self.fertility_raw / (self.fertility_raw.max() + 1e-9)
        
        # SVD coordinate projection
        U, Sigma, Vt = svds(self.X_csc.astype(np.float32, copy=False), k=svd_components, solver='arpack')
        self.U_sigma = U * Sigma
        self.Vt = Vt


def compute_ultimate_scores(cache, a, alpha, gamma, tau, lambda_svd, beta_fertility, zero_diag=True):
    """Generates a full 1D similarity profile matching the exact math of Version 7."""
    rows, cols = cache.rows, cache.cols
    N_smoothed = cache.N_raw + (alpha * rows * cols)
    
    col_a = cache.X_csc[:, a:a+1]
    curr_rows, curr_data = col_a.indices, col_a.data.astype(np.float32)
    
    P_x = (cache.row_sums_raw + (alpha * cols)) / N_smoothed
    P_y_a = (cache.col_sums_raw[a] + (alpha * rows)) / N_smoothed
    
    P_xy_direct = np.full(rows, alpha / N_smoothed, dtype=np.float32)
    P_xy_direct[curr_rows] = (curr_data + alpha) / N_smoothed
    
    if lambda_svd > 0:
        latent_profile = cache.U_sigma @ cache.Vt[:, a]
        
        # Restored high-fidelity Min-Max vector scaling
        latent_min, latent_max = latent_profile.min(), latent_profile.max()
        latent_profile = (latent_profile - latent_min) / (latent_max - latent_min + 1e-9)
        latent_profile = (latent_profile / (latent_profile.sum() + 1e-15)) * P_y_a
        
        P_xy_blended = (1.0 - lambda_svd) * P_xy_direct + lambda_svd * latent_profile
    else:
        P_xy_blended = P_xy_direct

    denominator = (P_x ** gamma) * (P_y_a ** gamma)
    if beta_fertility > 0:
        denominator *= (cache.fertility_norm ** beta_fertility)
        
    scores = P_xy_blended / (denominator + 1e-15)
    
    if tau > 0:
        raw_counts = np.zeros(rows, dtype=np.float32)
        raw_counts[curr_rows] = curr_data
        scores *= (raw_counts / (raw_counts + tau))
        
    if zero_diag:
        scores[a] = 0.0
    return scores


def optimize_ultimate_parameters(cache, item_groups, init_params=None, optimize_vars=None):
    """
    Optimizes configurations using the mathematically strict np.argsort 
    rank-inversion mapping to preserve true metric accuracy.
    """
    default_params = {'alpha': 0.1, 'gamma': 1.0, 'tau': 2.0, 'lambda_svd': 0.1, 'beta_fertility': 0.5}
    neutral_defaults = {'alpha': 0.0, 'gamma': 1.0, 'tau': 0.0, 'lambda_svd': 0.0, 'beta_fertility': 0.0}

    param_bounds = {
        'alpha': (0.0, 2.0),
        'gamma': (0.01, 3.0),
        'tau': (0.0, 25.0),
        'lambda_svd': (0.0, 0.95),
        'beta_fertility': (0.0, 3.0)
    }

    param_step_sizes = {
        'alpha': 0.02,
        'gamma': 0.1,
        'tau': 1.0,
        'lambda_svd': 0.02,      
        'beta_fertility': 0.1
    }

    if init_params is None:
        init_params = default_params.copy()
    else:
        for k, v in default_params.items():
            if k not in init_params:
                init_params[k] = v

    if optimize_vars is None:
        optimize_vars = ['alpha', 'gamma', 'tau', 'lambda_svd', 'beta_fertility']

    for var in ['alpha', 'gamma', 'tau', 'lambda_svd', 'beta_fertility']:
        if init_params[var] is None:
            assert var not in optimize_vars, f"AssertionError: Cannot optimize locked parameter '{var}' set to None."
            init_params[var] = neutral_defaults[var]

    unique_items = list(set(idx for group in item_groups for idx in group))
    x0 = [init_params[var] for var in optimize_vars]
    active_bounds = [param_bounds[var] for var in optimize_vars]
    
    n_vars = len(optimize_vars)
    init_direc = np.zeros((n_vars, n_vars))
    for i, var in enumerate(optimize_vars):
        init_direc[i, i] = param_step_sizes[var]

    def objective(x_vec):
        current_params = init_params.copy()
        for var_name, val in zip(optimize_vars, x_vec):
            current_params[var_name] = val
            
        a, g, t = current_params['alpha'], current_params['gamma'], current_params['tau']
        l_svd, b_fert = current_params['lambda_svd'], current_params['beta_fertility']
        
        if a < 0.0 or g <= 0.0 or t < 0.0 or l_svd < 0.0 or l_svd >= 0.95 or b_fert < 0.0:
            return 1e9
            
        # Generate strict full-profile scores
        current_scores = {idx: compute_ultimate_scores(cache, idx, a, g, t, l_svd, b_fert) for idx in unique_items}
        
        group_scores = []
        for group in item_groups:
            total_pairwise_rank = 0.0
            k = len(group)
            
            for u in group:
                sort_indices = np.argsort(-current_scores[u])
                ranks = np.empty_like(sort_indices)
                ranks[sort_indices] = np.arange(len(sort_indices))
                
                for v in group:
                    if u == v:
                        continue
                    total_pairwise_rank += ranks[v]
                    
            group_scores.append(total_pairwise_rank / (k * (k - 1)))
            
        return np.mean(group_scores)

    if len(optimize_vars) > 0:
        res = minimize(objective, x0, method='Powell', bounds=active_bounds, options={'direc': init_direc})
        final_values = res.x if len(optimize_vars) > 1 else [res.x]
        final_params = init_params.copy()
        for var_name, val in zip(optimize_vars, final_values):
            final_params[var_name] = float(val)
        return res, final_params
    else:
        return None, init_params

In [157]:
cache = SppmiUltimateCache(X, svd_components=32)

In [158]:
# terms = ["Megadeth"]
# terms = ["Havok (spotify:artist:2jw4wgixxa20jls9N3Bdpq)"]
# terms = ["Evile (spotify:artist:1dwrMJAKBiLlj0O4R791Xo)"]
# terms = ["Sonata Arctica"]
# terms = ["Muse (spotify:artist:12Chz98pHFMPJEknJQMWvI)"]
# terms = ["Dream Theater (spotify:artist:2aaLAng2L2aWD2FClzwiep)"]
# terms = ["Saints Go Machine"]
# terms = ["AURORA (spotify:artist:1WgXqy2Dd70QQOU7Ay074N)"]
# terms = ["Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)"]
# terms = ["Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)"]
# terms = ["Fleet Foxes"]
# terms = ["Vektor (spotify:artist:09mNj9XgCqgg6usfeXOoBg)"]
# terms = ["Dr. Living Dead (spotify:artist:0gLz6azFpZgHyJkJd5yuiM)"]
# terms = ["Lich King (spotify:artist:4rlxS0LeVnHz6z1zp2iJbz)"]
# terms = ["Skeletonwitch (spotify:artist:213mmq3zkNWx7CtfzftTC5)"]
# terms = ["Wintersun (spotify:artist:6ui6SwChan7c1KYBQCqGKV)"]
# terms = ["Chris Poland"]
# terms = ["Marty Friedman (spotify:artist:5czW6bitDSKbNBNDizRT9p)"]
# terms = ["Jason Becker (spotify:artist:0A4Z1qNp3lGWa9VI66M67D)"]
# terms = ["Cacophony (spotify:artist:3WNx4M2YbMmDiJqeOBi0Ae)"]

# a = get_item_idx(["Cacophony (spotify:artist:3WNx4M2YbMmDiJqeOBi0Ae)"])
# b = get_item_idx(["Jason Becker (spotify:artist:0A4Z1qNp3lGWa9VI66M67D)"])

In [159]:
item_group_a = [
    get_item_idx(["Highasakite"]),
    get_item_idx(["Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)"]),
]

Artist: Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)
********************
Artist: Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)
********************


In [160]:
item_group_b = [
    get_item_idx(["Cacophony (spotify:artist:3WNx4M2YbMmDiJqeOBi0Ae)"]),
    get_item_idx(["Jason Becker (spotify:artist:0A4Z1qNp3lGWa9VI66M67D)"]),
#     get_item_idx(["Marty Friedman (spotify:artist:5czW6bitDSKbNBNDizRT9p)"]),
]

Artist: Cacophony (spotify:artist:3WNx4M2YbMmDiJqeOBi0Ae)
********************
Artist: Jason Becker (spotify:artist:0A4Z1qNp3lGWa9VI66M67D)
********************


In [161]:
nested_items = [
    item_group_a,
    item_group_b,
]

In [162]:
# nested_items = [
#     item_group_a,
# ]

In [165]:
# 2. Run an experiment tuning all 5 dimensions simultaneously
res, best_params = optimize_ultimate_parameters(
    cache,
    item_groups=nested_items,
    init_params={
        'alpha': 0.06397197616493325, 'gamma': 1.2987089696156853, 'tau': 13.338553077819997,
#         'alpha': 0.05877717876369542, 'gamma': 1.4384908592804768, 'tau': 12.500107867164445,
        'lambda_svd': .1, 
        'beta_fertility': None
    },
    # Hierarchical order of Powell coordinate searches
    optimize_vars=['tau', 'gamma', 'alpha']
)

In [166]:
print("Optimized Parameters:", best_params)
print("Best Achieved Mean Rank:", res.fun)

Optimized Parameters: {'alpha': 0.06964840876960202, 'gamma': 1.4390467731431453, 'tau': 7.900662295103562, 'lambda_svd': 0.1, 'beta_fertility': 0.0}
Best Achieved Mean Rank: 3.0
